# Identification des paramètres thermiques par réseau convolutif temporel (TCN)

## Objectif de l'expérience

Cette expérience étudie un **réseau convolutif temporel (Temporal Convolutional Network, TCN)** pour estimer les propriétés thermiques d'un bâtiment à partir des signaux de température intérieure $T(t)$ et de sollicitation thermique $U(t)$.

L'architecture utilise des **convolutions dilatées et des connexions résiduelles** pour extraire des caractéristiques à différentes échelles temporelles et apprendre les dépendances associées aux phénomènes de déperdition et d'inertie thermique.

L'objectif est d'évaluer les performances du TCN et sa capacité à généraliser à des profils de sollicitation non rencontrés pendant l'entraînement, en comparaison avec les autres architectures étudiées.

## 1. Environnement d'exécution

Les fonctions de chargement des données, de définition des modèles, d'entraînement et d'évaluation sont regroupées dans `src/coenergy/` afin de faciliter la réutilisation du code et la reproductibilité des expériences.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import shutil
import sys

import torch
from IPython.display import display, FileLink


def find_source_directory():
    """Localise les modules Python du projet coEnergy."""

    current_directory = Path.cwd().resolve()

    for directory in (current_directory, *current_directory.parents):
        source_directory = directory / "src"
        if (source_directory / "coenergy" / "loss.py").is_file():
            return source_directory

    kaggle_directory = Path("/kaggle/input")

    if kaggle_directory.is_dir():
        module_files = list(kaggle_directory.rglob("src/coenergy/loss.py"))
        if module_files:
            return module_files[0].parent.parent

        archives = list(kaggle_directory.rglob("coenergy_modules_kaggle.zip"))
        if archives:
            destination = Path("/kaggle/working/coenergy_code")

            with ZipFile(archives[0]) as archive:
                archive.extractall(destination)

            source_directory = destination / "src"
            if (source_directory / "coenergy" / "loss.py").is_file():
                return source_directory

    raise FileNotFoundError(
        "Modules coEnergy introuvables : ajouter le dataset contenant "
        "coenergy_modules_kaggle.zip."
    )


source_directory = find_source_directory()
sys.path.insert(0, str(source_directory))

from coenergy.data import load_data, BatchStore
from coenergy.models import build_model
from coenergy.loss import family_loss
from coenergy.training import default_config, seed_everything, Experiment
from coenergy.evaluation import evaluate_all, extreme_diagnostics
from coenergy.plotting import plot_training, plot_scores, plot_scatter

print("Modules coEnergy chargés depuis :", source_directory)

## 2. Configuration expérimentale

L'expérience porte exclusivement sur le **TCN enrichi**. Ses hyperparamètres sont définis dans `training.py` : **120 époques au maximum**, **mini-batchs de 512 simulations**, optimisation **AdamW** et adaptation du taux d'apprentissage en fonction de la perte de validation.

Un **arrêt anticipé** est prévu après 22 époques consécutives sans amélioration suffisante de la validation. Les paramètres de référence sont conservés, notamment une graine aléatoire fixe et un dropout de 0,05. Un nouvel identifiant `RUN_ID` distingue une nouvelle expérience d'une reprise.

In [ ]:
MODEL_KIND = "tcn"
RUN_ID = "run_001"
DATA_ROOT = (
    Path("/kaggle/input/datasets/bilalaouiss/dataset")
    if Path("/kaggle/input/datasets/bilalaouiss/dataset").is_dir()
    else None
)
RUN_TRAINING = True
RESUME = True

config = default_config(MODEL_KIND, RUN_ID)
seed_everything(config.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if Path("/kaggle/working").is_dir():
    results_directory = Path("/kaggle/working/results")
else:
    results_directory = source_directory.parent / "results"

output_directory = results_directory / MODEL_KIND / RUN_ID

print("Modèle :", MODEL_KIND)
print("Appareil :", device)
print("Époques maximales :", config.max_epochs)
print("Taille des mini-batchs :", config.batch_size)
print("Patience de l'arrêt anticipé :", config.early_patience)
print("Résultats :", output_directory)

## 3. Données préparées et partitions

Le jeu de données comprend **1 000 000 de simulations**. Les quatre partitions, établies lors de l'analyse exploratoire, restent identiques pour l'ensemble des architectures.

| Partition | Simulations | Fonction |
|---|---:|---|
| Entraînement | 719 885 | Ajustement des paramètres du réseau |
| Validation | 89 985 | Sélection du checkpoint et adaptation du taux d'apprentissage |
| Test principal | 89 987 | Mesure des performances sur des simulations non utilisées pendant l'apprentissage |
| Réserve U | 100 143 | Évaluation sur des profils de sollicitation réservés |


In [ ]:
data = load_data(DATA_ROOT, expected_n=1_000_000)

for partition_name, indices in data.splits.items():
    print(f"{partition_name:12s} : {len(indices):,} simulations")

print("Séries temporelles :", data.sequences.shape)
print("Cibles thermiques :", data.targets.shape)

## 4. Préparation des signaux temporels

Chaque simulation contient **480 observations** de la température $T$ et de la sollicitation $U$. Pour un mini-batch de $B$ simulations, l'entrée possède la forme $(B,2,480)$.

Les transformations utilisent exclusivement les statistiques calculées sur la partition d'entraînement :

$$
\widetilde{T}=\frac{T-\mu_{T,\mathrm{train}}}{\sigma_{T,\mathrm{train}}},
\qquad
\widetilde{U}=\frac{U}{\mathrm{RMS}_{U,\mathrm{train}}}.
$$

La température est centrée et réduite, tandis que la sollicitation est divisée par sa valeur quadratique moyenne sans centrage. `BatchStore` applique ces transformations et prépare les mini-batchs.

**Les variations temporelles et les six statistiques complémentaires utilisées par le TCN sont calculées dans `models.py`**, à partir de ces signaux normalisés.

In [ ]:
training_data = BatchStore(
    bundle=data,
    mode="tcn",
    device=device,
)

sample_signals, sample_targets = training_data.get_batch(
    data.splits["train"][:8]
)

print("Entrées du TCN :", tuple(sample_signals.shape))
print("Cibles :", tuple(sample_targets.shape))

## 5. Architecture du TCN enrichi

Le modèle enrichit d'abord les deux signaux normalisés par leurs différences successives, $\Delta T$ et $\Delta U$. Il obtient ainsi **quatre canaux temporels** de 480 points. Six statistiques globales complètent la représentation : variation totale et amplitude de $T$, proportion de sollicitation active, moyenne de $U$ et variations absolues moyennes de $T$ et $U$.

Le réseau comporte **sept blocs résiduels**, chacun constitué de deux convolutions de noyau 3, avec les facteurs de dilatation $1,2,4,8,16,32,64$. Les dilatations augmentent le champ réceptif sans multiplier fortement le nombre de paramètres. Les connexions résiduelles facilitent la circulation de l'information et des gradients. Les convolutions utilisent un **padding symétrique** : l'architecture n'est donc pas causale.

Après l'encodeur, un **pooling moyen** et un **pooling maximum** produisent chacun 16 caractéristiques. Leur concaténation avec les six statistiques forme un vecteur de **38 valeurs**, transmis à une tête de régression $38\to64\to5$.

Les cinq sorties indépendantes sont rendues positives par **Softplus**. Le sixième indicateur est reconstruit par la relation physique :

$$
\widehat{HLC}=\widehat{HLC}_{ext}+\widehat{HLC}_{mit}.
$$

La définition des couches et l'initialisation des sorties se trouvent dans `models.py`.

In [ ]:
model = build_model(MODEL_KIND, data.prep).to(device)

number_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

model.eval()

with torch.no_grad():
    six_predictions = model(sample_signals)

print(model)
print("Paramètres entraînables :", f"{number_parameters:,}")
print("Sorties du TCN :", tuple(six_predictions.shape))
print(
    "Contrainte HLC respectée :",
    torch.allclose(
        six_predictions[:, 0],
        six_predictions[:, 1] + six_predictions[:, 2],
    ),
)

## 6. Fonction de perte

L'apprentissage porte sur les **cinq paramètres indépendants** : $HLC_{ext}$, $HLC_{mit}$, $\tau_b$, $\tau_n$ et $\tau_\infty$. Les erreurs sont divisées par les écarts-types correspondants du train pour éviter qu'une famille de cibles domine uniquement par son échelle numérique.

La perte combine les erreurs quadratiques moyennes normalisées des deux familles :

$$
\mathcal{L}=0{,}50\,\mathcal{L}_{HLC}+0{,}50\,\mathcal{L}_{\tau}.
$$

Le coefficient $HLC$ total est **reconstruit, mais non pénalisé séparément**, afin de ne pas compter deux fois la même information physique. Le TCN renvoyant six colonnes, la première est exclue du calcul de la perte.

In [ ]:
initial_loss = family_loss(
    six_predictions[:, 1:].float(),
    sample_targets,
    data.prep["target_std"],
    hlc_weight=config.hlc_weight,
    tau_weight=config.tau_weight,
)

print("Perte initiale :", round(initial_loss.item(), 6))
print("Poids des déperditions :", config.hlc_weight)
print("Poids des constantes de temps :", config.tau_weight)

## 7. Entraînement et sélection du modèle

Les poids du TCN sont mis à jour uniquement sur `train`. Après chaque époque, la perte est mesurée sur `validation` sans modification des paramètres. Elle sert à ajuster le taux d'apprentissage, à identifier la meilleure époque et à déclencher l'arrêt anticipé en cas de stagnation.

L'entraînement utilise **AdamW**, une limitation de la norme du gradient et la **précision mixte sur GPU**. Le fichier `history.csv` conserve les pertes et l'évolution du taux d'apprentissage.

Deux checkpoints sont enregistrés : `best.pt` contient les meilleurs poids selon la validation, tandis que `last.pt` conserve l'état nécessaire à la reprise. Si `RUN_TRAINING = False`, un dossier de résultats avec des checkpoints compatibles doit déjà être disponible.

In [ ]:
experiment = Experiment(
    config=config,
    bundle=data,
    store=training_data,
    model=model,
    out_dir=output_directory,
    device=device,
)

if RUN_TRAINING:
    history = experiment.fit(resume=RESUME)
    print("Époques réalisées :", len(history))
else:
    print("Entraînement désactivé : utilisation des checkpoints existants.")

## 8. Courbes d'apprentissage

L'évaluation finale repose sur **le meilleur checkpoint de validation**, et non nécessairement sur les poids de la dernière époque. Cette sélection ne dépend pas des ensembles de test.

Les courbes des pertes d'entraînement et de validation permettent d'analyser la convergence et les écarts de généralisation. Une seconde figure représente l'évolution du taux d'apprentissage.

In [ ]:
best_epoch = experiment.load_best()

print("Meilleure époque :", best_epoch)

plot_training(
    experiment.history_path,
    output_directory / "figures",
    best_epoch,
)

## 9. Évaluation des performances

Le checkpoint sélectionné est évalué sur `validation`, le **test principal** et la **réserve U**. Pour chacun des six indicateurs thermiques, l'évaluation calcule notamment :

- **MAE** : erreur absolue moyenne, dans l'unité du paramètre ;
- **RMSE** : racine de l'erreur quadratique moyenne, sensible aux grandes erreurs ;
- **$R^2$** : coefficient de détermination, mesurant la qualité des prédictions relativement à une référence constante définie sur la partition évaluée.

Une **baseline supplémentaire**, utilisant les moyennes des cibles du train, permet de comparer le TCN à une prédiction constante. Les ensembles de test restent exclus des décisions d'entraînement et de sélection.

In [ ]:
metrics = evaluate_all(experiment)

selected_metrics = metrics[
    (metrics["model"] == MODEL_KIND)
    & (metrics["partition"].isin(["test", "reserve_u"]))
]

display(
    selected_metrics[
        ["partition", "target", "unit", "MAE", "RMSE", "R2", "bias"]
    ].round(4)
)

plot_scores(metrics, MODEL_KIND, output_directory / "figures")

## 10. Analyse des prédictions et des valeurs extrêmes

Les diagrammes de dispersion confrontent les **valeurs réelles et prédites** sur le test principal ; la diagonale représente une prédiction parfaite.

Une analyse complémentaire examine les erreurs sur les valeurs extrêmes des constantes de temps $\tau_b$ et $\tau_\infty$. Les seuils sont fixés aux **5e et 95e percentiles des cibles du train**, puis appliqués au test principal et à la réserve U. Cette démarche permet de mesurer les erreurs sur des cas peu représentés sans utiliser les tests pour définir les seuils.

In [ ]:
plot_scatter(
    output_directory / "predictions" / "test.npz",
    output_directory / "figures",
    seed=config.seed,
)

extreme_metrics = extreme_diagnostics(experiment)
display(extreme_metrics.round(4))

## 11. Sauvegarde des résultats

Les modèles entraînés, les configurations, les historiques d'entraînement, les métriques d'évaluation, les prédictions et les figures sont enregistrés afin d'assurer la traçabilité et la reproductibilité des expériences.

L'ensemble des résultats est également regroupé dans une archive ZIP.

In [ ]:
if Path("/kaggle/working").is_dir():
    archive_directory = Path("/kaggle/working")
else:
    archive_directory = results_directory

archive_path = Path(
    shutil.make_archive(
        str(archive_directory / f"{MODEL_KIND}_coenergy"),
        "zip",
        root_dir=output_directory,
    )
)

print("Fichier trouvé :", archive_path.exists())
print("Taille :", round(archive_path.stat().st_size / 1024**2, 2), "Mo")

if Path.cwd().resolve() == archive_directory.resolve():
    display(FileLink(archive_path.name))
else:
    display(FileLink(str(archive_path)))


## 12. Analyse et interprétation des résultats

Le réseau convolutif temporel (TCN) repose sur des convolutions dilatées et des connexions résiduelles. Cette architecture permet d'étendre le champ réceptif et de traiter des dépendances temporelles éloignées sans utiliser de couche récurrente.

### Performances prédictives

| Paramètre | R² — Test | R² — Réserve U |
|---|---:|---:|
| HLC | 0.411 | 0.414 |
| HLCext | 0.434 | 0.430 |
| HLCmit | 0.268 | 0.269 |
| τb | 0.126 | 0.124 |
| τn | 0.254 | 0.247 |
| τ∞ | 0.099 | 0.098 |

Le TCN améliore les performances du MLP et du CNN 1D sur les six indicateurs. Il atteint notamment un R² de 0.434 pour HLCext et de 0.411 pour HLC.

Ces résultats confirment l'intérêt des convolutions temporelles dilatées pour représenter la réponse thermique. Le modèle reste également relativement stable sur les profils de sollicitation non vus.

Cependant, contrairement à la tendance observée lors des premières expérimentations sur 500 000 simulations, le TCN final ne dépasse pas le CNN-GRU sur le jeu de test du million. Par exemple, pour HLCext, le R² du TCN est de 0.434 contre 0.481 pour le CNN-GRU.

Cette différence ne permet pas de conclure à une supériorité générale des architectures récurrentes. Les modèles possèdent des capacités et des choix d'architecture différents, et une seule exécution par configuration ne suffit pas à établir une conclusion statistique.

### Conclusion

Le TCN constitue une alternative pertinente aux réseaux récurrents pour l'identification thermique. Ses performances montrent que les convolutions dilatées peuvent exploiter les dépendances temporelles, mais leur utilisation ne garantit pas une amélioration systématique par rapport au CNN-GRU.
